
# Merge XGBoost LOO Joblib Files → Single Joblib + HDF5

This notebook scans a local `data/` folder for files matching `*_loo_xgboost_LOO_results.joblib`, loads each (expected to be a pandas DataFrame), adds a `protein` column from the filename if missing, merges them, and saves:
- `xgb_all_results.joblib` — merged `DataFrame` serialized with joblib
- `xgb_all_results.h5` — HDF5; prefers pandas HDFStore if PyTables is available; otherwise falls back to a generic HDF5 with CSV bytes

> If you see an error about missing `tables`, install PyTables (`pip install tables` or `conda install pytables`) and re-run the **Save HDF5** cell.


In [1]:
# Cell 1 — setup and helpers
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)

import os, traceback
from pathlib import Path
import pandas as pd
import joblib

# Paths & settings
DATA_DIR = Path("data")
OUT_DIR  = Path("data_h5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PARTS   = 3
OVERWRITE = False  # set True to regenerate existing .h5 files

def coerce_to_df(obj):
    if isinstance(obj, pd.DataFrame):
        return obj
    if hasattr(obj, "to_dataframe"):
        try:
            return obj.to_dataframe()
        except Exception:
            pass
    try:
        return pd.DataFrame(obj)
    except Exception:
        return pd.DataFrame({"value": [obj]})

def process_subset(subset_id, overwrite=False):
    """Slice files into N_PARTS and write this subset to HDF5 (format='fixed')."""
    all_files = sorted(DATA_DIR.glob("*.joblib"))
    subset = all_files[subset_id::N_PARTS]

    print(f"Subset {subset_id+1}/{N_PARTS}: {len(subset)} files")
    ok = err = 0
    for i, path in enumerate(subset, 1):
        out_path = OUT_DIR / (path.stem + ".h5")
        if out_path.exists() and not overwrite:
            print(f"[{i}/{len(subset)}] SKIP (exists): {path.name}")
            continue
        try:
            obj = joblib.load(path)
            df  = coerce_to_df(obj)
            df.to_hdf(out_path, key="data", mode="w", format="fixed")
            ok += 1
            print(f"[{i}/{len(subset)}] {path.name} -> {out_path.name} (rows={len(df)})")
        except Exception:
            err += 1
            print(f"[{i}/{len(subset)}] ERROR: {path.name}")
            traceback.print_exc()
    print(f"Done subset {subset_id+1}/{N_PARTS}. OK={ok}, ERR={err}")


convert directly to h5

In [2]:
# Cell 2 — run subset 1/3
process_subset(0, overwrite=OVERWRITE)
# Cell 3 — run subset 2/3
process_subset(1, overwrite=OVERWRITE) 
# Cell 4 — run subset 3/3   
process_subset(2, overwrite=OVERWRITE)

Subset 1/3: 1705 files
[1/1705] A2M_loo_xgboost_LOO_results.joblib -> A2M_loo_xgboost_LOO_results.h5 (rows=369)
[2/1705] AAGAB_loo_xgboost_LOO_results.joblib -> AAGAB_loo_xgboost_LOO_results.h5 (rows=369)
[3/1705] AAMP_loo_xgboost_LOO_results.joblib -> AAMP_loo_xgboost_LOO_results.h5 (rows=369)
[4/1705] AARS_loo_xgboost_LOO_results.joblib -> AARS_loo_xgboost_LOO_results.h5 (rows=369)
[5/1705] AASS_loo_xgboost_LOO_results.joblib -> AASS_loo_xgboost_LOO_results.h5 (rows=369)
[6/1705] ABCB6_loo_xgboost_LOO_results.joblib -> ABCB6_loo_xgboost_LOO_results.h5 (rows=369)
[7/1705] ABCC1_loo_xgboost_LOO_results.joblib -> ABCC1_loo_xgboost_LOO_results.h5 (rows=369)
[8/1705] ABCD3_loo_xgboost_LOO_results.joblib -> ABCD3_loo_xgboost_LOO_results.h5 (rows=369)
[9/1705] ABCF2_loo_xgboost_LOO_results.joblib -> ABCF2_loo_xgboost_LOO_results.h5 (rows=369)
[10/1705] ABHD11_loo_xgboost_LOO_results.joblib -> ABHD11_loo_xgboost_LOO_results.h5 (rows=369)
[11/1705] ABHD16A_loo_xgboost_LOO_results.joblib -> AB

In [3]:
# Open a single HDF5 file from data_h5 and display its head
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_H5 = Path("data_h5")

# Set one of these:
fname = 'ZZZ3_loo_xgboost_LOO_results.h5'                  # e.g. "protein_0001.h5"  (if you know the file name)
file_index = 0                # or pick by index among sorted files

files = sorted(DATA_H5.glob("*.h5"))
if not files:
    raise FileNotFoundError(f"No .h5 files found in {DATA_H5.resolve()}")

path = (DATA_H5 / fname) if fname else files[file_index]
if not path.exists():
    raise FileNotFoundError(f"{path.resolve()} not found")

print(f"Opening: {path.name}")

# Discover keys and choose one (prefer 'data' if present)
with pd.HDFStore(path, mode="r") as store:
    keys = list(store.keys())
    if not keys:
        raise RuntimeError(f"No tables found in {path.name}")
    key = "/data" if "/data" in keys else keys[0]
    if key != "/data":
        print(f"Info: key 'data' not found. Using '{key.lstrip('/')}'")
    print(f"Available keys: {keys}")

df = pd.read_hdf(path, key=key.lstrip("/"))
display(df.head(20))  # show first 20 rows
print(f"Rows={len(df)}, Cols={df.shape[1]}, Key='{key.lstrip('/')}'")


Opening: ZZZ3_loo_xgboost_LOO_results.h5
Available keys: ['/data']


,protein,LOO sample,true,predicted,params_dict,features_coef_dict,features_selected
0,ZZZ3,MDAMB468_BREAST,-0.054057,0.109687,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 4.923404693603516, 'ZNF644': 0.290849...","[MTF2, ZNF644, RBMXL1, ZZZ3, AGL, PTBP2, NSUN4..."
1,ZZZ3,SH4_SKIN,-0.371537,-0.117819,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 3.618044853210449, 'ZNF644': 0.438067...","[MTF2, ZNF644, RBMXL1, ZZZ3, AGL, PTBP2, NSUN4..."
2,ZZZ3,AU565_BREAST,-0.301741,0.005007,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 4.156810760498047, 'RBMXL1': 1.904228...","[MTF2, RBMXL1, ZNF644, ZZZ3, AGL, PTBP2, NSUN4..."
3,ZZZ3,KMRC1_KIDNEY,-0.732279,-0.912197,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 3.707352638244629, 'ZNF644': 0.540854...","[MTF2, ZNF644, RBMXL1, ZZZ3, PTBP2, AGL, NSUN4..."
4,ZZZ3,CAL51_BREAST,0.697868,0.505344,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 4.246556282043457, 'RBMXL1': 2.549995...","[MTF2, RBMXL1, ZNF644, ZZZ3, AGL, PTBP2, NSUN4..."
5,ZZZ3,RPMI7951_SKIN,0.166516,-0.217314,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 5.319062232971191, 'ZNF644': 0.461714...","[MTF2, ZNF644, RBMXL1, ZZZ3, AGL, PTBP2, NSUN4..."
6,ZZZ3,RERFLCMS_LUNG,0.573529,-0.295328,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 3.6573727130889893, 'ZNF644': 0.31644...","[MTF2, ZNF644, RBMXL1, ZZZ3, PTBP2, AGL, YEATS..."
7,ZZZ3,IGR37_SKIN,0.177767,0.167166,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 3.9993269443511963, 'ZNF644': 0.37554...","[MTF2, ZNF644, RBMXL1, ZZZ3, AGL, PTBP2, NSUN4..."
8,ZZZ3,VMRCRCW_KIDNEY,-0.451014,-0.297996,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 4.042369842529297, 'ZNF644': 0.508466...","[MTF2, ZNF644, RBMXL1, ZZZ3, PTBP2, AGL, YEATS..."
9,ZZZ3,HEP3B217_LIVER,-0.354818,-0.181416,"{'max_depth': 6, 'min_child_weight': 1, 'gamma...","{'MTF2': 3.8557281494140625, 'ZNF644': 0.32275...","[MTF2, ZNF644, RBMXL1, ZZZ3, PTBP2, AGL, NSUN4..."


Rows=369, Cols=7, Key='data'


Remove params columns to lighten the file size

In [4]:
# Batch-trim HDF5 files: drop heavy columns and save to final_data/
from pathlib import Path
import pandas as pd
import traceback

SRC_DIR = Path("data_h5")
DST_DIR = Path("final_data")
DST_DIR.mkdir(parents=True, exist_ok=True)

PREFERRED_KEY = "data"
DROP_COLS = {"params_dict", "features_coef_dict", "features_selected"}

files = sorted(SRC_DIR.glob("*.h5"))
print(f"Found {len(files)} files in {SRC_DIR}")

for i, fp in enumerate(files, 1):
    try:
        # Pick key (prefer '/data', otherwise first key)
        with pd.HDFStore(fp, mode="r") as st:
            keys = list(st.keys())
            if not keys:
                print(f"[{i}/{len(files)}] WARNING: {fp.name} has no keys; skipping.")
                continue
            use_key = "/"+PREFERRED_KEY if "/"+PREFERRED_KEY in keys else keys[0]
            if use_key != "/"+PREFERRED_KEY:
                print(f"[{i}/{len(files)}] INFO: {fp.name}: using key '{use_key.lstrip('/')}'")

        # Read, drop columns if present
        df = pd.read_hdf(fp, key=use_key.lstrip("/"))
        before_shape = df.shape
        to_drop = [c for c in DROP_COLS if c in df.columns]
        if to_drop:
            df = df.drop(columns=to_drop)
        df = df.reset_index(drop=True)

        # Write trimmed file to destination (keep same key and filename)
        out_path = DST_DIR / fp.name
        df.to_hdf(out_path, key=use_key.lstrip("/"), mode="w", format="fixed")

        print(f"[{i}/{len(files)}] {fp.name} -> {out_path.name}  {before_shape} -> {df.shape}  dropped={to_drop}")

    except Exception as e:
        print(f"[{i}/{len(files)}] ERROR processing {fp.name}: {e}")
        traceback.print_exc()


Found 5113 files in data_h5
[1/5113] A2M_loo_xgboost_LOO_results.h5 -> A2M_loo_xgboost_LOO_results.h5  (369, 7) -> (369, 4)  dropped=['features_coef_dict', 'params_dict', 'features_selected']
[2/5113] AAAS_loo_xgboost_LOO_results.h5 -> AAAS_loo_xgboost_LOO_results.h5  (369, 7) -> (369, 4)  dropped=['features_coef_dict', 'params_dict', 'features_selected']
[3/5113] AACS_loo_xgboost_LOO_results.h5 -> AACS_loo_xgboost_LOO_results.h5  (369, 7) -> (369, 4)  dropped=['features_coef_dict', 'params_dict', 'features_selected']
[4/5113] AAGAB_loo_xgboost_LOO_results.h5 -> AAGAB_loo_xgboost_LOO_results.h5  (369, 7) -> (369, 4)  dropped=['features_coef_dict', 'params_dict', 'features_selected']
[5/5113] AAK1_loo_xgboost_LOO_results.h5 -> AAK1_loo_xgboost_LOO_results.h5  (369, 7) -> (369, 4)  dropped=['features_coef_dict', 'params_dict', 'features_selected']
[6/5113] AAMDC_loo_xgboost_LOO_results.h5 -> AAMDC_loo_xgboost_LOO_results.h5  (369, 7) -> (369, 4)  dropped=['features_coef_dict', 'params_di

In [5]:
# Merge all per-protein HDF5 files into a single HDF5 (robust)
from pathlib import Path
import pandas as pd
import numpy as np
import traceback
import warnings
from pandas.errors import PerformanceWarning
import gc

# Quiet PyTables' object-pickling warnings (object columns are expected)
warnings.filterwarnings("ignore", category=PerformanceWarning)

DATA_H5  = Path("final_data")
OUT_PATH = Path("all_data.h5")   # change if we want a different location/name
PREFERRED_KEY = "data"           # we try this first; fall back to the first key if missing

# Optional: keep only specific columns to reduce memory (set to a list or leave as None)
KEEP_ONLY = None
# e.g. KEEP_ONLY = ["protein", "LOO sample", "true", "predicted"]

files = sorted(DATA_H5.glob("*.h5"))
print(f"Found {len(files)} h5 files in {DATA_H5}")

frames = []
all_cols = []   # running union of column names (keeps stable order)

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure string, unique, flat column names (handles MultiIndex, duplicates)."""
    cols = df.columns
    if isinstance(cols, pd.MultiIndex):
        new_cols = ["__".join(map(str, tup)) for tup in cols.to_list()]
    else:
        new_cols = [str(c) for c in cols.to_list()]

    # De-duplicate while preserving order (suffix with .1, .2, ...)
    seen = {}
    uniq_cols = []
    for c in new_cols:
        if c not in seen:
            seen[c] = 0
            uniq_cols.append(c)
        else:
            seen[c] += 1
            uniq_cols.append(f"{c}.{seen[c]}")
    df.columns = uniq_cols
    return df

def _coerce_keep_only(df: pd.DataFrame, keep):
    if keep is None:
        return df
    # Keep the intersection and preserve keep order
    safe_keep = [c for c in keep if c in df.columns]
    return df.loc[:, safe_keep] if safe_keep else df.iloc[:, 0:0]  # empty frame with 0 cols if none matched

def _update_union_columns(current_union, new_cols):
    """Append columns not yet in union, preserving order."""
    for c in new_cols:
        if c not in current_union:
            current_union.append(c)

# Load all frames, being tolerant to missing keys or odd structures
for i, fp in enumerate(files, 1):
    try:
        # Open store, pick a key
        with pd.HDFStore(fp, mode="r") as store:
            keys = list(store.keys())
            if not keys:
                print(f"[{i}/{len(files)}] WARNING: {fp.name} contains no keys; skipping.")
                continue

            if f"/{PREFERRED_KEY}" in keys:
                key_to_use = PREFERRED_KEY
            else:
                # Fall back to the first available key (strip the leading "/")
                key_to_use = keys[0].lstrip("/")
                if key_to_use != PREFERRED_KEY:
                    print(f"[{i}/{len(files)}] INFO: {fp.name} missing key '{PREFERRED_KEY}', using '{key_to_use}'")

        # Now actually read the selected key
        df = pd.read_hdf(fp, key=key_to_use)

        # Ensure flat string columns and no dupes
        df = _flatten_columns(df)

        # Normalize index (avoid concat alignment surprises)
        df = df.reset_index(drop=True)

        # Optionally reduce columns early
        df = _coerce_keep_only(df, KEEP_ONLY)

        # Track union of columns to align later
        _update_union_columns(all_cols, list(df.columns))

        frames.append(df)

        if i % 50 == 0 or i == len(files):
            print(f"[{i}/{len(files)}] loaded: {fp.name} (rows={len(df)}, cols={df.shape[1]})")

    except (OSError, ValueError, KeyError) as e:
        print(f"[{i}/{len(files)}] ERROR reading {fp.name}: {e}")
        traceback.print_exc()
    except Exception as e:
        print(f"[{i}/{len(files)}] UNEXPECTED ERROR reading {fp.name}: {e}")
        traceback.print_exc()

if not frames:
    raise RuntimeError("No data frames loaded — nothing to merge.")

# Align columns across all frames (outer-join semantics with NaNs for missing)
aligned = []
for idx, f in enumerate(frames, 1):
    # Reindex to union; keep types as-is (pandas will upcast as needed)
    af = f.reindex(columns=all_cols)
    aligned.append(af)
    # Periodic GC for large merges
    if idx % 100 == 0:
        gc.collect()

# Concatenate
merged = pd.concat(aligned, axis=0, ignore_index=True, sort=False, copy=False)
print(f"Merged shape: {merged.shape}")

# Write as 'fixed' (pickle-friendly; supports object columns)
# If this ever raises due to a pathological object type, uncomment the astype('object') fallback below.
# merged = merged.astype('object')
merged.to_hdf(OUT_PATH, key=PREFERRED_KEY, mode="w", format="fixed")
print(f"Wrote: {OUT_PATH.resolve()}  (rows={len(merged)}, cols={merged.shape[1]})")


Found 5113 h5 files in final_data
[50/5113] loaded: ACBD5_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[100/5113] loaded: ADNP_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[150/5113] loaded: AKAP8L_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[200/5113] loaded: ANKLE2_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[250/5113] loaded: APP_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[300/5113] loaded: ARL3_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[350/5113] loaded: ATF7IP_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[400/5113] loaded: ATPAF1_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[450/5113] loaded: BCL2L13_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[500/5113] loaded: BST2_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[550/5113] loaded: C3orf17_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[600/5113] loaded: CARKD_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[650/5113] loaded: CCDC50_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[700/5113] loaded: CDC20_loo_xgboost_LOO

In [ ]:
# Open the merged HDF5 file and display its head + NaN counts for true/predicted
from pathlib import Path
import pandas as pd
from IPython.display import display

h5_path = Path("xgb_all_results.h5")
preferred_key = "data"

if not h5_path.exists():
    raise FileNotFoundError(f"{h5_path.resolve()} not found")

with pd.HDFStore(h5_path, mode="r") as store:
    keys = list(store.keys())
    if not keys:
        raise RuntimeError(f"No tables found in {h5_path}")
    use_key = f"/{preferred_key}" if f"/{preferred_key}" in keys else keys[0]
    if use_key != f"/{preferred_key}":
        print(f"Info: key '{preferred_key}' not found. Using '{use_key.lstrip('/')}'")

df = pd.read_hdf(h5_path, key=use_key.lstrip("/"))
display(df.head())
print(f"Rows={len(df)}, Cols={df.shape[1]}, Key='{use_key.lstrip('/')}'")

# NaN counts (and percentages) in true and predicted
if {"true", "predicted"}.issubset(df.columns):
    n_true_nan = int(df["true"].isna().sum())
    n_pred_nan = int(df["predicted"].isna().sum())
    n_either_nan = int(df[["true","predicted"]].isna().any(axis=1).sum())
    total = len(df) if len(df) > 0 else 1
    print(
        f"NaNs — true: {n_true_nan} ({n_true_nan/total:.2%}), "
        f"predicted: {n_pred_nan} ({n_pred_nan/total:.2%}), "
        f"either: {n_either_nan} ({n_either_nan/total:.2%})"
    )
else:
    print("Columns 'true' and/or 'predicted' not found. Available columns:", list(df.columns))


,protein,LOO sample,true,predicted
0,A2M,MDAMB468_BREAST,-0.345562,-0.691980
1,A2M,SH4_SKIN,1.642186,0.469850
2,A2M,AU565_BREAST,-1.129114,-0.811306
3,A2M,KMRC1_KIDNEY,1.493333,0.888209
4,A2M,CAL51_BREAST,0.274652,0.562785


Rows=1886697, Cols=4, Key='data'
NaNs — true: 0 (0.00%), predicted: 381546 (20.22%), either: 381546 (20.22%)


In [7]:
# Cell 1 — proteins that contain NaNs in true or predicted
import pandas as pd
from IPython.display import display

assert {"protein","true","predicted"}.issubset(df.columns), "df must have columns: protein, true, predicted"

nan_rows = df[["true","predicted"]].isna().any(axis=1)
nan_subset = df.loc[nan_rows, ["protein","true","predicted"]].copy()
nan_subset["nan_true"] = nan_subset["true"].isna()
nan_subset["nan_pred"] = nan_subset["predicted"].isna()

total_by_prot = df.groupby("protein").size().rename("total_rows")
nan_summary = (
    nan_subset.groupby("protein")
    .agg(n_nan_rows=("protein","size"),
         n_nan_true=("nan_true","sum"),
         n_nan_pred=("nan_pred","sum"))
    .join(total_by_prot, how="left")
    .assign(pct_nan=lambda x: x["n_nan_rows"] / x["total_rows"])
    .sort_values(["n_nan_rows","pct_nan"], ascending=[False,False])
)

print(f"Proteins with at least one NaN: {nan_summary.shape[0]}")
display(nan_summary.head(30))   # top 30 by NaN count
nan_summary.index.to_series().to_csv("proteins_with_nans.txt", index=False, header=False)
print("Saved: proteins_with_nans.txt")


Proteins with at least one NaN: 1034


,n_nan_rows,n_nan_true,n_nan_pred,total_rows,pct_nan
protein,,,,,
CAD,369,0,369,369,1.0
CALD1,369,0,369,369,1.0
CALR,369,0,369,369,1.0
CAMK2D,369,0,369,369,1.0
CAMSAP2,369,0,369,369,1.0
CANX,369,0,369,369,1.0
CAPN1,369,0,369,369,1.0
CAPNS1,369,0,369,369,1.0
CAPZA2,369,0,369,369,1.0


Saved: proteins_with_nans.txt


In [8]:
# Cell — compare proteins in file vs data (handles your path & formatting)
from pathlib import Path
import pandas as pd

# Build set of proteins in your loaded data (normalize whitespace & case)
assert "protein" in df.columns, "Expected column 'protein' in df"
proteins_in_data = (
    df["protein"]
    .dropna().astype(str).str.strip().str.upper()
    .unique().tolist()
)
proteins_in_data = set(proteins_in_data)

# Try common locations; picks the first that exists
candidates = [
    Path("protein_list.txt"),
    Path("model_results") / "protein_list.txt",
    Path("04. xgb") / "model_results" / "protein_list.txt",  # from your screenshot
]
for cand in candidates:
    if cand.exists():
        list_path = cand
        break
else:
    raise FileNotFoundError(
        "protein_list.txt not found in any of: "
        + ", ".join(str(p) for p in candidates)
    )

# Read file: ignore blank lines and lines starting with '#'
with list_path.open("r", encoding="utf-8") as f:
    file_lines = [ln.strip() for ln in f]

file_series = (
    pd.Series(file_lines, dtype="string")
      .str.strip()
      .pipe(lambda s: s[~s.str.startswith("#", na=False)])
      .replace("", pd.NA)
      .dropna()
      .str.upper()
      .drop_duplicates()
)

file_proteins = set(file_series.tolist())

missing = sorted(file_proteins - proteins_in_data)   # in file but NOT in data
extras  = sorted(proteins_in_data - file_proteins)   # in data but NOT in file
overlap = sorted(file_proteins & proteins_in_data)

print(f"File: {list_path}  |  total in file (unique): {len(file_proteins)}")
print(f"In file BUT NOT in data: {len(missing)}")
print(missing[:50])  # preview
print(f"\nIn data BUT NOT in file: {len(extras)}")
print(extras[:50])   # preview
print(f"\nOverlap: {len(overlap)}")

print("\nSaved: proteins_not_in_all_data.txt, proteins_in_data_not_in_list.txt")


File: protein_list.txt  |  total in file (unique): 5113
In file BUT NOT in data: 0
[]

In data BUT NOT in file: 0
[]

Overlap: 5113

Saved: proteins_not_in_all_data.txt, proteins_in_data_not_in_list.txt
